# Data preparation and evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Syndr0/compute-efficient-genqa/blob/main/notebooks/00_data.ipynb)

In [ ]:
import os
import sys

import torch

USE_DRIVE = True
SMOKE_TEST = False
GPU_PROFILE = "auto"  # auto | t4 | a100

REPO_DIR = "/content/compute-efficient-genqa"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Syndr0/compute-efficient-genqa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -e .
sys.path.insert(0, f"{REPO_DIR}/src")

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/cegqa_outputs"
else:
    OUT = "/content/cegqa_outputs"

GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
if GPU_PROFILE == "auto":
    GPU_PROFILE = "a100" if GPU_MEM_GB >= 39 else "t4"
CONFIG_FILES = ["configs/default.yaml"]
if GPU_PROFILE == "a100":
    CONFIG_FILES.append("configs/a100.yaml")
    OUT += "_a100"
if SMOKE_TEST:
    CONFIG_FILES.append("configs/smoke.yaml")
    OUT += "_smoke"
os.environ["CEGQA_OUTPUT_DIR"] = OUT
CONFIG = " ".join(f"--config {f}" for f in CONFIG_FILES)
print(f"GPU: {GPU} ({GPU_MEM_GB:.0f} GB) | profile: {GPU_PROFILE} | outputs: {OUT}")

In [ ]:
!python -m cegqa.cli prepare $CONFIG

In [ ]:
import json

import pandas as pd
from IPython.display import Image, Markdown, display

from cegqa.data import eval_sources
from cegqa.utils import load_config, read_jsonl

cfg = load_config(CONFIG_FILES)

In [ ]:
rep = json.load(open(f"{cfg['data_dir']}/data_report.json"))
rows = []
for split in ("train", "dev", "test"):
    for src, v in rep[split].items():
        rows.append({"split": split, "source": src, "n": v["n"],
                     "ctx_words": round(v["context_words"]["mean"]), "ctx_words_p90": v["context_words"]["p90"],
                     "answer_words": round(v["answer_words"]["mean"], 2),
                     "answer_in_context": round(100 * v["answer_in_context_rate"], 1)})
pd.DataFrame(rows)

In [ ]:
display(pd.DataFrame(rep["overlap_before_filter"]).T)
display(pd.DataFrame(rep["overlap_after_filter"]).T)

In [ ]:
!python -m pytest -q tests